# CV16 — Hán OCR voting on Kaggle
Enable **Internet**, attach a dataset containing `01_nguon/Cap_01/HAN.pdf` and `01_nguon/Cap_02/HAN.pdf`, then **Run All** in a fresh session. Optionally attach `02_ocr_tho/Cap_XX/HAN/images/NNNN/NNNN-CC.png` to reuse your numbered vertical strips. Strips must be upright with characters read top to bottom and numbered in page reading order (right to left). Pages without strips use the original notebook's cutter.
The default is a six-page pilot. PaddleOCR and Tesseract use CPU; EasyOCR uses Kaggle's GPU if available. No GPU is required. First-time model downloads take time. The last cell creates a ZIP under `/kaggle/working`. Inspect the review CSV and crop images before expanding the run.

In [ ]:
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
# Set these only to override automatic discovery or add verified references.
PDF_ROOT = None  # directory containing Cap_XX/HAN.pdf (normally 01_nguon)
CROP_ROOT = None  # directory containing Cap_XX/HAN/images/ (normally 02_ocr_tho)
REFERENCE_ROOT = None  # optional directory containing Cap_XX/HAN/NNNN.txt
PAGES = {'Cap_01': '1,27,53', 'Cap_02': '1,46,92'}
# Example: {'Cap_01': '1-10', 'Cap_02': [1, 3, 4]}; 'all' selects a whole volume.
FORCE_CPU = False  # applies to EasyOCR; PaddleOCR always uses CPU in this notebook
RENDER_DPI, PAGE_HEIGHT_FRACTION, MAX_SIDE = 144, 0.92, 2000
FORCE_SPINE_GAP = True  # original cutter; turn off if inspection shows clipped strips
ENGINE_ORDER = ('paddle', 'easyocr', 'tesseract')  # deterministic tie priority


In [ ]:
# Kaggle setup: use the session's PyTorch and avoid changing its CUDA stack.
import os
import shutil
import subprocess
import sys

if not INPUT.is_dir() or not WORK.is_dir():
    raise RuntimeError('Run this notebook in an internet-enabled Kaggle session with the data attached.')

packages = ['paddlepaddle==3.2.2', 'paddleocr==3.3.2', 'easyocr==1.7.2', 'PyMuPDF', 'scipy']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *packages], check=True)

def tesseract_languages():
    if not shutil.which('tesseract'):
        return set()
    proc = subprocess.run(['tesseract', '--list-langs'], capture_output=True, text=True, check=True)
    return set(proc.stdout.splitlines())

if 'chi_tra_vert' not in tesseract_languages():
    apt = [] if os.geteuid() == 0 else ['sudo', '-n']
    subprocess.run([*apt, 'apt-get', 'update', '-qq'], check=True)
    subprocess.run([*apt, 'apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-chi-tra-vert'], check=True)
if 'chi_tra_vert' not in tesseract_languages():
    raise RuntimeError('Tesseract cannot find chi_tra_vert. Check the setup output and TESSDATA_PREFIX.')
os.environ.setdefault('OMP_THREAD_LIMIT', '1')
os.environ.setdefault('PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK', 'True')
print('Setup complete.')


In [ ]:
import csv
import json
import re
import time
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from uuid import uuid4

import cv2
import numpy as np
import pymupdf
from scipy.ndimage import median_filter

def parse_pages(spec, count):
    if spec == 'all':
        return list(range(1, count + 1))
    tokens = spec.split(',') if isinstance(spec, str) else spec
    pages = set()
    for token in tokens:
        match = re.fullmatch(r'(\d+)(?:\s*-\s*(\d+))?', str(token).strip())
        if not match:
            raise ValueError(f'Invalid page or range: {token!r}')
        first = int(match[1])
        last = int(match[2] or match[1])
        if not 1 <= first <= last <= count:
            raise ValueError(f'Invalid range {first}-{last}; this PDF has {count} pages.')
        pages.update(range(first, last + 1))
    if not pages:
        raise ValueError('Select at least one page per volume.')
    return sorted(pages)

def locate_sources():
    if not PAGES or any(not re.fullmatch(r'Cap_\d{2}', cap) for cap in PAGES):
        raise ValueError('Use volume names such as Cap_01 in PAGES.')
    if PDF_ROOT is None:
        candidates = sorted({p for p in INPUT.rglob('01_nguon') if p.is_dir()})
        roots = [p for p in candidates if all((p / cap / 'HAN.pdf').is_file() for cap in PAGES)]
        if len(roots) != 1:
            raise ValueError(f'Expected one 01_nguon with the selected PDFs, found {roots}. Set PDF_ROOT above.')
        pdf_root = roots[0]
    else:
        pdf_root = Path(PDF_ROOT)
    crop_roots = ([Path(CROP_ROOT)] if CROP_ROOT is not None else
                  sorted({p for p in INPUT.rglob('02_ocr_tho') if p.is_dir()}))
    for root in crop_roots:
        if not root.is_dir():
            raise FileNotFoundError(f'CROP_ROOT does not exist: {root}')
    if REFERENCE_ROOT is not None and not Path(REFERENCE_ROOT).is_dir():
        raise FileNotFoundError(f'REFERENCE_ROOT does not exist: {REFERENCE_ROOT}')
    volumes = {}
    for cap, spec in sorted(PAGES.items()):
        pdf = pdf_root / cap / 'HAN.pdf'
        if not pdf.is_file():
            raise FileNotFoundError(pdf)
        with pymupdf.open(pdf) as doc:
            volumes[cap] = (pdf, parse_pages(spec, doc.page_count))
    return volumes, crop_roots

VOLUMES, CROP_ROOTS = locate_sources()
print({cap: pages for cap, (_, pages) in VOLUMES.items()})
print('Crop roots:', CROP_ROOTS or 'none; use the notebook cutter')

def preprocess_page(rgb):
    gray = cv2.cvtColor(rgb[:int(rgb.shape[0] * PAGE_HEIGHT_FRACTION)], cv2.COLOR_RGB2GRAY)
    gray = cv2.fastNlMeansDenoising(gray, None, h=10, templateWindowSize=7, searchWindowSize=21)
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    h, w = gray.shape
    scale = MAX_SIDE / max(h, w)
    if scale != 1:
        gray = cv2.resize(gray, (max(1, int(w * scale)), max(1, int(h * scale))),
                          interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    return gray

def extract_columns(gray):
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    projection = median_filter(binary.sum(axis=0), size=9)
    if not projection.max():
        return []
    gaps = projection < projection.max() * 0.10
    w = gray.shape[1]
    lo, hi = int(w * 0.40), int(w * 0.60)
    if FORCE_SPINE_GAP and lo < hi:
        spine = lo + int(projection[lo:hi].argmax())
        gaps[max(0, spine - 20):min(w, spine + 20)] = True
    columns, start = [], None
    for end, is_gap in enumerate(np.r_[gaps, True]):
        if not is_gap and start is None:
            start = end
        elif is_gap and start is not None:
            width = end - start
            if width > 150 and end < w:
                mid = start + width // 2
                lo, hi = max(start, mid - 40), min(end, mid + 40)
                split = lo + int(projection[lo:hi].argmin())
                columns.extend((gray[:, max(0, start - 10):min(w, split + 10)],
                                gray[:, max(0, split - 10):min(w, end + 10)]))
            elif width > 20:
                columns.append(gray[:, max(0, start - 15):min(w, end + 15)])
            start = None
    return columns[::-1]

def supplied_crops(cap, page_id):
    matches = []
    for root in CROP_ROOTS:
        directory = root / cap / 'HAN' / 'images' / page_id
        files = sorted(p for p in directory.glob(f'{page_id}-*') if p.is_file() and p.suffix.lower() == '.png')
        if files:
            matches.append((directory, files))
    if not matches:
        return [], []
    if len(matches) > 1:
        raise ValueError(f'{cap}/{page_id}: multiple crop datasets; set CROP_ROOT.')
    paths = {}
    for path in matches[0][1]:
        match = re.fullmatch(rf'{page_id}-(\d{{2,}})\.png', path.name, re.IGNORECASE)
        if not match or int(match[1]) < 1 or int(match[1]) in paths:
            raise ValueError(f'Malformed or duplicate crop number: {path}')
        paths[int(match[1])] = path
    numbers = sorted(paths)
    warnings = [] if numbers == list(range(1, numbers[-1] + 1)) else ['noncontiguous_crop_numbers']
    return [(i, paths[i]) for i in numbers], warnings

def prepare_page(cap, doc, page_no, image_dir):
    page_id = f'{page_no:04d}'
    supplied, warnings = supplied_crops(cap, page_id)
    image_dir.mkdir(parents=True, exist_ok=True)
    pix = doc[page_no - 1].get_pixmap(dpi=RENDER_DPI, colorspace=pymupdf.csRGB, alpha=False)
    pix.save(image_dir / f'{page_id}.png')
    crops = []
    if supplied:
        for i, src in supplied:
            dst = image_dir / f'{page_id}-{i:02d}.png'
            shutil.copyfile(src, dst)
            crops.append(dst)
        return crops, str(supplied[0][1].parent), warnings
    rgb = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    for i, gray in enumerate(extract_columns(preprocess_page(rgb)), 1):
        dst = image_dir / f'{page_id}-{i:02d}.png'
        if not cv2.imwrite(str(dst), gray):
            raise OSError(f'Could not write {dst}')
        crops.append(dst)
    return crops, 'generated_from_pdf', warnings


In [ ]:
# Engines use the same upright strips. Bounding boxes are sorted top to bottom.
def init_paddle():
    from paddleocr import PaddleOCR
    model = PaddleOCR(lang='chinese_cht', ocr_version='PP-OCRv5', device='cpu',
                      enable_mkldnn=False, cpu_threads=2,
                      use_doc_orientation_classify=False, use_doc_unwarping=False,
                      use_textline_orientation=True, text_det_limit_side_len=1600,
                      text_det_limit_type='max', text_det_box_thresh=0.3,
                      text_det_thresh=0.2, text_det_unclip_ratio=1.5, text_rec_score_thresh=0.0)

    def recognize(path):
        chunks = []
        for result in model.predict(str(path)):
            data = result.json['res']
            for box, text, score in zip(data['rec_polys'], data['rec_texts'], data['rec_scores'], strict=True):
                chunks.append({'box': box, 'text': text, 'score': float(score)})
        chunks.sort(key=top_y)
        return ''.join(item['text'] for item in chunks), chunks

    return recognize, {'device': 'cpu', 'lang': 'chinese_cht', 'ocr_version': 'PP-OCRv5',
                       'detector': 'PP-OCRv5_server_det', 'recognizer': 'PP-OCRv5_server_rec'}

def top_y(item):
    return min(float(point[1]) for point in item['box'])

def init_easyocr():
    import easyocr
    import torch
    gpu = not FORCE_CPU and torch.cuda.is_available()
    reader = easyocr.Reader(['ch_tra', 'en'], gpu=gpu, verbose=False)

    def recognize(path):
        chunks = []
        for box, text, score in reader.readtext(str(path), detail=1, paragraph=False, workers=0):
            chunks.append({'box': [[float(x), float(y)] for x, y in box],
                           'text': text, 'score': float(score)})
        chunks.sort(key=top_y)
        return ''.join(item['text'] for item in chunks), chunks

    return recognize, {'device': str(reader.device), 'lang': ['ch_tra', 'en']}

def init_tesseract():
    def recognize(path):
        proc = subprocess.run(['tesseract', str(path), 'stdout', '-l', 'chi_tra_vert', '--psm', '5'],
                              capture_output=True, encoding='utf-8', errors='replace', timeout=120, check=True)
        return proc.stdout, {'stderr': proc.stderr}

    return recognize, {'device': 'cpu', 'lang': 'chi_tra_vert', 'psm': 5,
                       'version': subprocess.check_output(['tesseract', '--version'], text=True).splitlines()[0]}

def normalize(text):
    # Keep punctuation and traditional/variant character distinctions.
    return ''.join(c for c in unicodedata.normalize('NFC', text) if not c.isspace() and c != '\ufeff')

def run_engine(recognize, init_error, path):
    start = time.perf_counter()
    if recognize is None:
        return {'status': 'unavailable', 'text': '', 'seconds': 0, 'error': init_error}
    try:
        raw, detail = recognize(path)
        text = normalize(raw)
        return {'status': 'ok' if text else 'empty', 'text': text, 'raw_text': raw,
                'seconds': time.perf_counter() - start, 'detail': detail}
    except Exception as exc:
        return {'status': 'error', 'text': '', 'seconds': time.perf_counter() - start,
                'error': f'{type(exc).__name__}: {exc}'}


In [ ]:
# Center-star voting: choose the observed string with minimum total edit distance,
# then align each other string to it. Empty/failed engines abstain.
def edit_distance(a, b):
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        current = [i]
        for j, y in enumerate(b, 1):
            current.append(min(previous[j] + 1, current[-1] + 1, previous[j - 1] + (x != y)))
        previous = current
    return previous[-1]

def project_to_anchor(anchor, text):
    n, m = len(anchor), len(text)
    d = [list(range(m + 1))] + [[i] + [0] * m for i in range(1, n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i - 1][j - 1] + (anchor[i - 1] != text[j - 1]),
                          d[i - 1][j] + 1, d[i][j - 1] + 1)
    aligned, inserts = [''] * n, [[] for _ in range(n + 1)]
    i, j = n, m
    while i or j:
        if i and j and d[i][j] == d[i - 1][j - 1] + (anchor[i - 1] != text[j - 1]):
            aligned[i - 1] = text[j - 1]
            i, j = i - 1, j - 1
        elif i and d[i][j] == d[i - 1][j] + 1:
            i -= 1
        else:
            inserts[i].append(text[j - 1])
            j -= 1
    return aligned, [''.join(reversed(chunk)) for chunk in inserts]

def vote(results):
    texts = {key: results[key]['text'] for key in ENGINE_ORDER if results[key]['text']}
    if not texts:
        return {'text': '', 'anchor': None, 'voters': [], 'flags': ['no_text'], 'decisions': []}
    anchor_key = min(texts, key=lambda key: sum(edit_distance(texts[key], s) for s in texts.values()))
    anchor = texts[anchor_key]
    projected = {key: project_to_anchor(anchor, value) for key, value in texts.items()}
    pieces, decisions = [], []

    def choose(values, fallback, kind, index):
        value, count = Counter(values.values()).most_common(1)[0]
        majority = count * 2 > len(values)
        chosen = value if majority else fallback
        if len(set(values.values())) > 1:
            decisions.append({'type': kind, 'anchor_index': index, 'candidates': values,
                              'chosen': chosen, 'rule': 'majority' if majority else 'anchor_fallback'})
        pieces.append(chosen)

    for i in range(len(anchor) + 1):
        # Competing insertion blocks are compared as units; an empty string is a gap.
        choose({k: v[1][i] for k, v in projected.items()}, '', 'before_anchor_index', i)
        if i < len(anchor):
            choose({k: v[0][i] for k, v in projected.items()}, anchor[i], 'character', i)
    flags = []
    if len(texts) < len(ENGINE_ORDER):
        flags.append('single_engine' if len(texts) == 1 else 'missing_engine_text')
    if decisions:
        flags.append('disagreement')
    if any(d['rule'] == 'anchor_fallback' for d in decisions):
        flags.append('anchor_fallback')
    return {'text': ''.join(pieces), 'anchor': anchor_key, 'voters': list(texts),
            'flags': flags, 'decisions': decisions}


In [ ]:
# Run the pilot and save results page by page. Re-running creates a fresh directory.
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:6]
RUN = WORK / f'cv16_han_ocr_{run_id}'
RUN.mkdir(exist_ok=False)

def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def write_csv(path, fields, rows):
    with path.open('w', newline='', encoding='utf-8-sig') as out:
        writer = csv.DictWriter(out, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

adapters, engines = {}, {}
initializers = {'paddle': init_paddle, 'easyocr': init_easyocr, 'tesseract': init_tesseract}
for key in ENGINE_ORDER:
    start = time.perf_counter()
    try:
        adapters[key], config = initializers[key]()
        engines[key] = {'status': 'ready', 'config': config}
    except Exception as exc:
        engines[key] = {'status': 'unavailable', 'error': f'{type(exc).__name__}: {exc}'}
    engines[key]['init_seconds'] = time.perf_counter() - start
    print(key, engines[key])

variants = (*ENGINE_ORDER, 'ensemble')
page_rows, review_rows, cer_rows, reference_issues = [], [], [], []
reference_pages = []
for cap, (pdf, pages) in VOLUMES.items():
    with pymupdf.open(pdf) as doc:
        for page_no in pages:
            page_id = f'{page_no:04d}'
            base = RUN / '02_ocr_tho' / cap / 'HAN'
            image_dir = base / 'images' / page_id
            raw = base / 'raw' / page_id
            reports, texts, coverage, elapsed = [], {k: [] for k in variants}, Counter(), Counter()
            errors, warnings, source = Counter(), [], ''
            start_page = time.perf_counter()
            page_error = ''
            try:
                strips, source, warnings = prepare_page(cap, doc, page_no, image_dir)
                if not strips:
                    page_error = 'No usable crops found; inspect the page image.'
            except Exception as exc:
                strips = []
                page_error = f'{type(exc).__name__}: {exc}'

            for path in strips:
                results = {k: run_engine(adapters.get(k), engines[k].get('error'), path) for k in ENGINE_ORDER}
                start_vote = time.perf_counter()
                ensemble = vote(results)
                elapsed['ensemble'] += time.perf_counter() - start_vote
                texts['ensemble'].append(ensemble['text'])
                coverage['ensemble'] += bool(ensemble['text'])
                for key, result in results.items():
                    texts[key].append(result['text'])
                    coverage[key] += bool(result['text'])
                    elapsed[key] += result['seconds']
                    errors[key] += result['status'] in ('error', 'unavailable')
                    write_json(raw / key / f'{path.stem}.json', result)
                reports.append({'strip': path.stem, **ensemble})
                flags = ensemble['flags'] + warnings
                if flags:
                    review_rows.append({'cap': cap, 'page': page_id, 'strip': path.stem,
                                        'flags': ';'.join(flags), 'ensemble': ensemble['text'],
                                        **{k: results[k]['text'] for k in ENGINE_ORDER}})

            joined = {k: ''.join(texts[k]) for k in variants}
            reviewed_strips = sum(bool(item['flags']) for item in reports)
            status = ('error' if page_error or not joined['ensemble'] else
                      'review' if warnings or reviewed_strips else 'ok')
            if not page_error and not joined['ensemble']:
                page_error = 'No ensemble text; inspect the engine results.'
            if page_error:
                review_rows.append({'cap': cap, 'page': page_id, 'strip': '', 'flags': page_error})
            write_json(raw / 'ensemble' / 'review.json', {'crop_source': source, 'warnings': warnings,
                                                        'error': page_error, 'strips': reports})
            for key in variants:
                (raw / key).mkdir(parents=True, exist_ok=True)
                (raw / key / f'{page_id}.txt').write_text(joined[key], encoding='utf-8')
                page_rows.append({'cap': cap, 'page': page_id, 'variant': key, 'crop_source': source,
                                  'strips': len(strips), 'nonempty_strips': coverage[key],
                                  'failed_strips': (len(strips) - coverage[key] if key == 'ensemble' else errors[key]),
                                  'seconds': round(elapsed[key], 3), 'status': status,
                                  'flags': ';'.join(warnings), 'error': page_error})

            ref = Path(REFERENCE_ROOT) / cap / 'HAN' / f'{page_id}.txt' if REFERENCE_ROOT else None
            if ref and ref.is_file():
                try:
                    truth = normalize(ref.read_text(encoding='utf-8-sig'))
                    if not truth:
                        raise ValueError('Reference text is empty.')
                except (OSError, UnicodeError, ValueError) as exc:
                    reference_issues.append({'cap': cap, 'page': page_id, 'error': str(exc)})
                else:
                    reference_pages.append((cap, page_id))
                    # A successful but empty OCR result counts as deletions, not a skipped sample.
                    for key in variants:
                        usable = bool(strips) and (sum(errors.values()) < len(strips) * len(ENGINE_ORDER)
                                                   if key == 'ensemble' else errors[key] == 0)
                        edits = edit_distance(truth, joined[key]) if usable else None
                        cer_rows.append({'cap': cap, 'page': page_id, 'variant': key,
                                         'reference_chars': len(truth), 'edits': edits,
                                         'cer': edits / len(truth) if edits is not None else None})
            print(f'{cap}/{page_id}: {status}; {len(strips)} strips, {reviewed_strips} flagged, '
                  f'{time.perf_counter() - start_page:.1f}s', page_error)

write_csv(RUN / 'pages.csv', ['cap', 'page', 'variant', 'crop_source', 'strips', 'nonempty_strips',
                             'failed_strips', 'seconds', 'status', 'flags', 'error'], page_rows)
write_csv(RUN / 'review.csv', ['cap', 'page', 'strip', 'flags', 'ensemble', *ENGINE_ORDER], review_rows)
write_csv(RUN / 'cer.csv', ['cap', 'page', 'variant', 'reference_chars', 'edits', 'cer'], cer_rows)

# Compare only identical labelled pages scored by every variant; never rank by agreement.
eligible = Counter((r['cap'], r['page']) for r in cer_rows if r['edits'] is not None)
common = [p for p in reference_pages if eligible[p] == len(variants)]
common_set = set(common)
totals = {}
for key in variants:
    rows = [r for r in cer_rows if r['variant'] == key and (r['cap'], r['page']) in common_set]
    if rows:
        totals[key] = sum(r['edits'] for r in rows) / sum(r['reference_chars'] for r in rows)
leaders = [key for key in totals if totals[key] == min(totals.values())]
manifest = {'run_id': run_id, 'pdfs': {k: str(v[0]) for k, v in VOLUMES.items()},
            'pages': {k: v[1] for k, v in VOLUMES.items()}, 'crop_roots': list(map(str, CROP_ROOTS)),
            'render_dpi': RENDER_DPI, 'crop_height_fraction': PAGE_HEIGHT_FRACTION,
            'max_side': MAX_SIDE, 'force_spine_gap': FORCE_SPINE_GAP, 'engines': engines,
            'packages': {name: version(name) for name in ('paddlepaddle', 'paddleocr', 'paddlex', 'easyocr',
                        'torch', 'torchvision', 'PyMuPDF', 'scipy', 'numpy')},
            'opencv': cv2.__version__, 'python': sys.version, 'tie_priority': list(ENGINE_ORDER),
            'reference_pages': reference_pages, 'reference_issues': reference_issues,
            'common_scored_pages': common, 'common_page_cer': totals,
            'lowest_cer_on_common_pages': leaders}
write_json(RUN / 'manifest.json', manifest)
if not leaders:
    print('Best model undetermined: no verified pages scored by all variants.')
else:
    print('Lowest CER on the common reference pages:', leaders, totals)


In [ ]:
from IPython.display import FileLink, display

archive = Path(shutil.make_archive(str(RUN), 'zip', root_dir=RUN.parent, base_dir=RUN.name))
print(f'Download from Kaggle Output: {archive}')
display(FileLink(os.path.relpath(archive)))
